# LAB-HW-06 — Real PS/Linux ↔ PL Loopback

**One new thing today:** software running on the KV260 PS writes a real PL register path and reads a transformed value back.

Prerequisites: LSN-015 and LAB-HW-05.

**Project Trace:** RMD-012B · T-HW-006/T-HW-011

## 1. Freeze the semantic contract first

Lesson 15 modeled:

**Roundtrip:** `write value → PL stores/processes → read back result`

This Lab makes it physical with one deliberately boring transform:

`read = (write + 1) mod 2^32`

Do not add neurons, FIFO, DDR, or performance measurement yet.

## 2. The real transport path

<svg xmlns="http://www.w3.org/2000/svg" width="980" height="250" viewBox="0 0 980 250" role="img" aria-label="LAB-HW-06 PS Linux to PL MMIO loopback">
  <rect x="20" y="75" width="165" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="102" y="108" text-anchor="middle" font-size="14">Ubuntu / Python</text>
  <text x="102" y="132" text-anchor="middle" font-size="12">/dev/mem mmap</text>
  <rect x="225" y="75" width="160" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="305" y="108" text-anchor="middle" font-size="14">PS HPM0 FPD</text>
  <text x="305" y="132" text-anchor="middle" font-size="12">memory-mapped master</text>
  <rect x="425" y="75" width="145" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="498" y="108" text-anchor="middle" font-size="14">SmartConnect</text>
  <text x="498" y="132" text-anchor="middle" font-size="12">platform adapter</text>
  <rect x="610" y="75" width="160" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="690" y="105" text-anchor="middle" font-size="14">AXI GPIO</text>
  <text x="690" y="128" text-anchor="middle" font-size="12">0xA0010000</text>
  <text x="690" y="148" text-anchor="middle" font-size="11">+0 write / +8 read</text>
  <rect x="810" y="75" width="150" height="85" rx="10" fill="#e6f4ea" stroke="#333"/>
  <text x="885" y="108" text-anchor="middle" font-size="14">PL transform</text>
  <text x="885" y="132" text-anchor="middle" font-size="12">value + 1 mod 2^32</text>
  <path d="M185 117 L225 117 M385 117 L425 117 M570 117 L610 117 M770 117 L810 117" stroke="#333" stroke-width="2"/>
  <polygon points="225,117 215,112 215,122" fill="#333"/><polygon points="425,117 415,112 415,122" fill="#333"/>
  <polygon points="610,117 600,112 600,122" fill="#333"/><polygon points="810,117 800,112 800,122" fill="#333"/>
</svg>

You only need to understand the **layer boundaries** today. AXI channel details remain deferred.

The base address is frozen to `0xA0010000`. This matches AMD/Xilinx's K26 `base_gpio_bram` reference design. The course helper does not expose an arbitrary physical-address option.

## 3. Two register offsets

AMD PG144 defines AXI GPIO data registers:

| purpose | register | offset |
|---|---|---:|
| host writes value | Channel 1 `GPIO_DATA` | `0x0000` |
| host reads PL result | Channel 2 `GPIO2_DATA` | `0x0008` |

Channel 1 is configured as a 32-bit output. Channel 2 is configured as a 32-bit input.

The learner does **not** write an AXI slave in this Lab.

## 4. Check the oracle without hardware

On the development host:

```bash
python boards/kv260/runtime/loopback_mmio.py --dry-run
```

The output must end with `STATUS=PASS`.

This tests only the semantic oracle and host checker. It is **not** T-HW-006 physical evidence.

## 5. Build the dedicated loopback bitstream

On the development host:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-06-build.log \
  -source boards/kv260/scripts/build_lab06_loopback.tcl
```

The design uses:

- PS `M_AXI_HPM0_FPD`;
- AXI SmartConnect;
- dual-channel AXI GPIO at `0xA0010000`;
- `kv260_loopback_transform`;
- PS `pl_clk0` and synchronized reset.

The helper requires setup/hold timing closure before writing:

`build/kv260/lab-hw-06/kv260_loopback.bit`

## 6. Keep Linux running while programming PL

LAB-HW-05 Linux must already be booted.

On the **runtime host**, if a Kria app is active:

```bash
sudo xmutil unloadapp
```

Record the result. Do not treat “no app loaded” as an error by itself.

Then, on the **development host**:

```bash
vivado -mode batch -nojournal \
  -log lab-hw-06-program.log \
  -source boards/kv260/scripts/program_bitstream.tcl \
  -tclargs build/kv260/lab-hw-06/kv260_loopback.bit
```

Do **not** power-cycle after programming; the PL configuration would be lost.

## 7. Put the checker on the runtime host

The graded concept is MMIO loopback, not networking.

Copy `boards/kv260/runtime/loopback_mmio.py` to the PS by any already-working file-transfer method. If Ethernet is already available, one convenient option from the development host is:

```bash
scp boards/kv260/runtime/loopback_mmio.py ubuntu@<kv260-ip>:/tmp/
```

If networking is not configured, use another simple file-copy method. Do not turn this Lab into a network-debugging exercise.

## 8. Run the physical self-check

On PS/Linux:

```bash
sudo python3 /tmp/loopback_mmio.py \
  --json-out /tmp/lab-hw-06-trace.json
```

The default checker runs two rounds of the fixed vector set, including zero, normal values, and 32-bit wraparound.

Every line records:

`WRITE → EXPECTED → READ`

Final physical success requires `STATUS=PASS`.

## 9. Failure classes matter

The checker separates:

- `TRANSPORT_DEVICE_MISSING` — no `/dev/mem`;
- `TRANSPORT_REQUIRES_ROOT` — the physical check was not started with `sudo`;
- `TRANSPORT_PERMISSION_OR_POLICY` — OS policy/permission boundary;
- `TRANSPORT_MMAP_FAILED` — mapping/transport failure;
- `CORE_BEHAVIOR_MISMATCH` — MMIO worked, but the value returned from PL is wrong.

`/dev/mem` is the frozen **teaching transport for this Lab**, not the promised long-term MOD-010 software interface. If Ubuntu policy blocks it, **do not weaken system security to force the Lab to pass**. Save the evidence; T-HW-006 remains blocked and the transport must be revised in a later repository change.

## 10. Expected Evidence / Save Evidence

Retain:

- `lab-hw-06-build.log`, timing/resource/DRC reports;
- bitstream SHA-256;
- `lab-hw-06-program.log`;
- MMIO base `0xA0010000`, offsets `0x0` and `0x8`;
- SHA-256 of `loopback_mmio.py`;
- complete checker stdout;
- `lab-hw-06-trace.json`;
- Ubuntu image identity, kernel/OS, board/carrier revision;
- Git commit/date.

A dry-run PASS never substitutes for a physical MMIO PASS.

## 11. If it does not work

1. Linux not booted → return to LAB-HW-05.
2. Vivado cannot build → inspect IP/board-part/timing evidence.
3. JTAG cannot program → return to the LAB-HW-02/03 target path.
4. Linux disappears after PL programming → preserve logs; this is a platform/deployment failure, not a loopback mismatch.
5. `/dev/mem` permission/policy failure → record and stop; do not weaken security settings.
6. MMIO maps but readback is wrong → verify base address, offsets, bitstream hash, and transform RTL.

## 12. Human Check

Explain:

1. Which instructions execute on the PS CPU, and which behavior occurs in PL?
2. Why does `0xA0010000` matter?
3. Why are `+0x0` and `+0x8` different?
4. Why is AXI GPIO a teaching adapter rather than the FlyBrain algorithm?
5. Why can a transport-policy failure not be called a core-logic failure?
6. Why is `--dry-run` useful but insufficient?

## 13. Official basis

- AMD PG144 AXI GPIO — Channel 1 `GPIO_DATA` offset 0x0; Channel 2 `GPIO2_DATA` offset 0x8
- AMD/Xilinx `kria-base-hardware` K26 `base_gpio_bram/scripts/config_bd.tcl` — PS HPM0/SmartConnect/AXI GPIO reference and AXI GPIO address `0xA0010000`
- AMD Kria Ubuntu documentation and xmutil documentation

The hardware path and fixed-address teaching transport are frozen for LAB-HW-06. A real Ubuntu 24.04 KV260 run is still required before T-HW-006 can be marked physically passed.